<a href="https://colab.research.google.com/github/melonmeg/RAG_Medical_Assistant/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-openai langchain-huggingface faiss-cpu sentence-transformers pypdf unstructured


In [ ]:
import os

os.environ["GROQ_API_KEY"] = input("enter GROQ API KEY")


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
import os

documents = []

for file in os.listdir():
    if file.endswith(".pdf"):
        print(f"Loading {file}")

        loader = PyPDFLoader(file)
        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks")

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k":5}
)

results = retriever.invoke(
    "recommended treatment for hypertension in diabetic patients"
)

"""for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(doc.page_content[:500])"""

In [ ]:
"""docs = retriever.invoke(query)

for d in docs[:5]:
    print(d.page_content)"""

In [ ]:
pip install langchain_groq

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template( """
You are a clinical research assistant.


Retrieved Evidence:
{context}

Question:
{question}

Instructions:

1. Summarize findings.
2. Cite evidence.
3. Mention contradictions.
4. List treatment options.
5. Mention risks.
6. Provide confidence level.
7. Never hallucinate.
""")


In [ ]:
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {
        "context": itemgetter("question") | retriever ,
        "question": itemgetter("question"),

    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
def ask(question, history):

    return rag_chain.invoke({
        "question": question,
        "history": history
    })


In [ ]:
import gradio as gr

def respond(
    age,
    sex,
    weight,
    chief_complaint,
    symptom_duration,
    medical_history,
    medications,
    allergies,
    vitals,
    lab_results,
    question,
    chat_history,
):

    try:

        patient_history = f"""
Age: {age}
Sex: {sex}
Weight: {weight}

Chief Complaint:
{chief_complaint}

Duration:
{symptom_duration}

Medical History:
{medical_history}

Current Medications:
{medications}

Allergies:
{allergies}

Vitals:
{vitals}

Labs:
{lab_results}
"""

        history_text = ""

        response = ask(question, patient_history)

        chat_history.append((question, response))

        return "", chat_history

    except Exception as e:
        import traceback
        traceback.print_exc()
        raise e


with gr.Blocks(title="Clinical Research Assistant") as demo:

    gr.Markdown("# 🩺 Clinical Research Assistant")
    gr.Markdown("Enter the patient's clinical information before asking your question.")

    with gr.Row():

        with gr.Column(scale=1):

            gr.Markdown("## Patient History")

            age = gr.Number(
                label="Age",
                precision=0
            )

            sex = gr.Dropdown(
                ["Male", "Female", "Other"],
                label="Sex"
            )

            weight = gr.Number(
                label="Weight (kg)"
            )

            chief_complaint = gr.Textbox(
                label="Chief Complaint",
                lines=2,
                placeholder="e.g. Chest pain, fever, cough..."
            )

            symptom_duration = gr.Textbox(
                label="Duration / Onset",
                placeholder="e.g. Started 3 days ago"
            )

            medical_history = gr.Textbox(
                label="Past Medical History",
                lines=4,
                placeholder="Hypertension, Type 2 Diabetes..."
            )

            medications = gr.Textbox(
                label="Current Medications",
                lines=3,
                placeholder="Metformin, Aspirin..."
            )

            allergies = gr.Textbox(
                label="Drug Allergies",
                placeholder="Penicillin"
            )

            vitals = gr.Textbox(
                label="Vital Signs",
                lines=3,
                placeholder="""BP: 130/80
HR: 72
Temp: 98.6°F
SpO₂: 98%"""
            )

            lab_results = gr.Textbox(
                label="Laboratory Results",
                lines=4,
                placeholder="""HbA1c: 8.2%
Creatinine: 1.3
eGFR: 62"""
            )

        with gr.Column(scale=2):

            chatbot = gr.Chatbot(
                label="Clinical Assistant",
                height=550
            )

            question = gr.Textbox(
                label="Clinical Question",
                placeholder="Ask a question about this patient..."
            )

            submit = gr.Button(
                "Generate Evidence-Based Recommendation",
                variant="primary"
            )

    submit.click(
        respond,
        inputs=[
            age,
            sex,
            weight,
            chief_complaint,
            symptom_duration,
            medical_history,
            medications,
            allergies,
            vitals,
            lab_results,
            question,
            chatbot
        ],
        outputs=[
            question,
            chatbot
        ]
    )

    question.submit(
        respond,
        inputs=[
            age,
            sex,
            weight,
            chief_complaint,
            symptom_duration,
            medical_history,
            medications,
            allergies,
            vitals,
            lab_results,
            question,
            chatbot
        ],
        outputs=[
            question,
            chatbot
        ]
    )

demo.launch()